# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source

The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Get the schema metadata as an object
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview

Let's review the available record sets, fields, and their `@id`s in the dataset.

In [ ]:
# List record sets from the metadata
print("Available record sets in this dataset:")
record_sets = meta.record_sets
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '(no name)')}")

# For each record set, list the available fields (columns) by @id
for rs in record_sets:
    print(f"\nFields for RecordSet '{rs['@id']}' ({rs.get('name', '(no name)')}):")
    fields = rs.get('fields', [])
    for f in fields:
        print(f"  - {f['@id']} : {f.get('name', '(no name)')}")

## 3. Data Extraction

Load data from each record set into a DataFrame for analysis. We'll use the record set and field `@id`s from the overview above.

In [ ]:
# Get all record set @ids for extraction
record_set_ids = [rs['@id'] for rs in meta.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading records for RecordSet '@id': {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"{rs_id}: columns = {df.columns.tolist()}")
    else:
        print(f"{rs_id}: No records found.")

# Display the head of the main record set, if any found
if dataframes:
    main_rs_id = next(iter(dataframes))
    print(f"\nPreview of the first record set ({main_rs_id}):")
    display(dataframes[main_rs_id].head())
else:
    print("No record sets with data found in this dataset.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, and grouping data by key attributes. We will reference all fields by their `@id`.

> **Note:** Adjust `numeric_field_id` and `group_field_id` according to record set and field `@id` as printed above.

In [ ]:
# Pick the primary record set for EDA
if not dataframes:
    print("No data loaded to analyze.")
else:
    # Use the first available dataframe/record set as example
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id]

    # List numeric fields to select from
    print("Numeric fields detected (by @id):")
    numeric_fields = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
    if not numeric_fields:
        print("  (no numeric fields detected; will attempt to coerce numeric columns)")
        # Try to coerce all columns to numeric and see what's possible
        potential_numeric = [col for col in df.columns if pd.to_numeric(df[col], errors='coerce').notnull().any()]
        print("Potential numeric columns by @id:", potential_numeric)
        if potential_numeric:
            numeric_field = potential_numeric[0]
        else:
            print("No numeric field found; skipping EDA.")
            numeric_field = None
    else:
        print(numeric_fields)
        numeric_field = numeric_fields[0]

    # Proceed only if a numeric field is found
    if numeric_field:
        print(f"\nUsing numeric field: {numeric_field}")
        # Coerce to numeric
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        # Filter, e.g., threshold at 10, but this may be adjusted according to field's nature
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"\nFiltered records where {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nAdded normalized column: {normalized_col}")
        display(filtered_df[[numeric_field, normalized_col]].head())

        # Pick a group-by field (categorical)
        possible_group_fields = [col for col in df.columns if col != numeric_field and df[col].nunique() > 1 and df[col].nunique() < 10]
        if possible_group_fields:
            group_field = possible_group_fields[0]
            print(f"\nGrouping by: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(f"mean_{numeric_field}")
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

> Note: All fields are referenced by their `@id`. Adjust as appropriate for your dataset.

In [ ]:
import matplotlib.pyplot as plt

# Ensure we have data and a numeric field from previous steps
if 'df' in locals() and 'numeric_field' in locals() and numeric_field:
    # Histogram of the numeric field
    plt.figure(figsize=(8,4))
    df[numeric_field].hist(bins=15, edgecolor='black')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If we found a group_field, plot boxplot
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8,4))
        df.boxplot(column=numeric_field, by=group_field, grid=False)
        plt.title(f"{numeric_field} by {group_field}")
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No suitable data for visualization.")

## 6. Conclusion

In this notebook, we demonstrated how to load and explore the FAIR² dataset using the `mlcroissant` library.

Key steps covered:
* Loading metadata and data records directly from a Croissant schema URL
* Listing available record sets and fields by their unique `@id`
* Loading records into pandas DataFrames and performing EDA (filtering, normalization, grouping)
* Visualizing numeric attribute distributions and categorical relationships

This notebook provides a template for exploring any Croissant-conformant dataset. To extend this analysis, adjust the record set and field `@id`s as needed and explore richer statistical or machine learning workflows on this cohort!